# Stage 6 — Decision Tree Model Development

## Objective

Develop and evaluate a Decision Tree classifier for wildfire prediction.

The model will be trained using the 2013–2020 training data and evaluated
using the 2021–2022 validation data. The 2023–2025 test data will remain
untouched until final evaluation.

Class imbalance will be handled using class weighting because wildfire
observations represent a minority class in the dataset.

In [22]:
import pandas as pd

# Load the common Stage 6 datasets
train = pd.read_csv("../../data/processed/train.csv")
validation = pd.read_csv("../../data/processed/validation.csv")

In [23]:
# Separate input features and target

X_train = train.drop(columns=["wildfire"])
y_train = train["wildfire"]

X_validation = validation.drop(columns=["wildfire"])
y_validation = validation["wildfire"]

In [24]:
print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("Validation features:", X_validation.shape)
print("Validation target:", y_validation.shape)

Training features: (7319483, 20)
Training target: (7319483,)
Validation features: (1326842, 20)
Validation target: (1326842,)


In [25]:
print(y_train.value_counts())

wildfire
0    7045069
1     274414
Name: count, dtype: int64


In [26]:

# check for non-numeric columns in the training set
print(X_train.dtypes)

latitude                        float64
longitude                       float64
precipitation                   float64
relative_humidity_max           float64
relative_humidity_min           float64
specific_humidity               float64
solar_radiation                 float64
temperature_min                 float64
temperature_max                 float64
wind_speed                      float64
burning_index                   float64
fuel_moisture_100hr             float64
fuel_moisture_1000hr            float64
energy_release_component        float64
reference_evapotranspiration    float64
potential_evapotranspiration    float64
vapor_pressure_deficit          float64
year                              int64
month                             int64
day_of_year                       int64
dtype: object


In [27]:
from sklearn.tree import DecisionTreeClassifier

In [28]:
# Create a Decision Tree Classifier model with Gini impurity criterion and balanced class weights
dt_model = DecisionTreeClassifier(
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [29]:
#
dt_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_f

In [30]:
print(dt_model)

DecisionTreeClassifier(class_weight='balanced', random_state=42)


## Phase 6.4 — Baseline Decision Tree Validation

### Objective

Evaluate the baseline Decision Tree on the unseen validation dataset
(2021–2022) to measure how well the model generalizes beyond the training data.

In [31]:
# Generate class predictions for the validation dataset

y_pred = dt_model.predict(X_validation)

In [32]:
# Check how many predictions were generated

print("Number of validation predictions:", len(y_pred))

# Display the first 10 predictions

print("First 10 predictions:", y_pred[:10])

Number of validation predictions: 1326842
First 10 predictions: [0 0 0 0 0 0 0 0 0 0]


In [33]:
# Import classification evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Calculate validation performance metrics

accuracy = accuracy_score(y_validation, y_pred)

precision = precision_score(y_validation, y_pred)

recall = recall_score(y_validation, y_pred)

f1 = f1_score(y_validation, y_pred)

In [34]:
# Display validation performance

print("Baseline Decision Tree - Validation Performance")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Baseline Decision Tree - Validation Performance
Accuracy : 0.8956
Precision: 0.1235
Recall   : 0.0460
F1-score : 0.0670


In [35]:
# Import confusion matrix function

from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix

cm = confusion_matrix(y_validation, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1183313   35301]
 [ 103253    4975]]


In [36]:
# Generate probability predictions for the positive class (wildfire)
# [:, 1] selects the probability of class 1 = wildfire

y_prob = dt_model.predict_proba(
    X_validation
)[:, 1]

print("Probability predictions generated.")
print("Number of predictions:", len(y_prob))

Probability predictions generated.
Number of predictions: 1326842


In [37]:
# Import AUC metrics

from sklearn.metrics import roc_auc_score, average_precision_score

# Calculate ROC-AUC

roc_auc = roc_auc_score(
    y_validation,
    y_prob
)

# Calculate PR-AUC

pr_auc = average_precision_score(
    y_validation,
    y_prob
)

In [38]:
# Display AUC metrics

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC : {pr_auc:.4f}")

ROC-AUC: 0.5085
PR-AUC : 0.0834


In [39]:
# Store baseline validation results for later comparison

baseline_results = {
    "Model": "Decision Tree",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "ROC-AUC": roc_auc,
    "PR-AUC": pr_auc
}

print(baseline_results)

{'Model': 'Decision Tree', 'Accuracy': 0.8955761123027459, 'Precision': 0.12352269341543351, 'Recall': 0.04596777174113908, 'F1': 0.06700156224748155, 'ROC-AUC': 0.5084908104679842, 'PR-AUC': 0.08343022565538116}


In [40]:
print(cm)

[[1183313   35301]
 [ 103253    4975]]


## Phase 6.5 — Experiment Recording and Baseline Analysis

### Objective

Record the baseline Decision Tree configuration, validation results, and
important observations so that the model can be compared fairly with other
models and with the optimized Decision Tree in Stage 7.

In [41]:
# Record the baseline Decision Tree configuration

baseline_config = {
    "Model": "Decision Tree",
    "Criterion": "Gini",
    "Class Weight": "Balanced",
    "Random State": 42
}

print(baseline_config)

{'Model': 'Decision Tree', 'Criterion': 'Gini', 'Class Weight': 'Balanced', 'Random State': 42}


In [42]:
# Convert baseline results into a table for easy comparison later

baseline_results_df = pd.DataFrame(
    [baseline_results]
)

baseline_results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Decision Tree,0.895576,0.123523,0.045968,0.067002,0.508491,0.08343


In [43]:
# Save the baseline Decision Tree validation results

baseline_results_df.to_csv(
    "../../ml/results/decision_tree_baseline.csv",
    index=False
)

In [44]:
# Save the baseline confusion matrix

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Wildfire", "Actual Wildfire"],
    columns=["Predicted No Wildfire", "Predicted Wildfire"]
)

cm_df

,Predicted No Wildfire,Predicted Wildfire
Actual No Wildfire,1183313,35301
Actual Wildfire,103253,4975


In [45]:
# Save the confusion matrix for documentation

cm_df.to_csv(
    "../../ml/results/decision_tree_baseline_confusion_matrix.csv"
)

# Stage 7 — Decision Tree Model Optimization

## Phase 7.1 — Hyperparameter Understanding

### Objective

Identify the important Decision Tree hyperparameters that control model
complexity and performance.

The baseline Decision Tree established the initial validation performance.
In this stage, different hyperparameter configurations will be investigated
to determine whether the model can achieve better validation performance.

The validation dataset will be used to evaluate different configurations,
while the final test dataset will remain untouched until final evaluation.

In [46]:
# Display the current baseline Decision Tree parameters

dt_model.get_params()

{'ccp_alpha': 0.0,
 'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 42,
 'splitter': 'best'}

## Phase 7.2 — Hyperparameter Experiment Design

### Objective

Investigate whether controlling Decision Tree complexity can improve
validation performance compared with the baseline model.

## Phase 7.3 — Experiment 1: max_depth

### Objective

Investigate whether limiting the maximum depth of the Decision Tree
improves its ability to generalize to unseen validation data.

In [47]:
from sklearn.tree import DecisionTreeClassifier

# Create Decision Tree with a maximum depth of 10

dt_depth10 = DecisionTreeClassifier(
    max_depth=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [48]:
# Train the max_depth=10 Decision Tree using the training data

dt_depth10.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_fea

In [49]:
# Generate predictions for the validation dataset

y_pred_depth10 = dt_depth10.predict(X_validation)

In [50]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_depth10 = dt_depth10.predict_proba(
    X_validation
)[:, 1]

In [51]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Calculate validation metrics for max_depth=10

accuracy_depth10 = accuracy_score(
    y_validation,
    y_pred_depth10
)

precision_depth10 = precision_score(
    y_validation,
    y_pred_depth10
)

recall_depth10 = recall_score(
    y_validation,
    y_pred_depth10
)

f1_depth10 = f1_score(
    y_validation,
    y_pred_depth10
)

roc_auc_depth10 = roc_auc_score(
    y_validation,
    y_prob_depth10
)

pr_auc_depth10 = average_precision_score(
    y_validation,
    y_prob_depth10
)

In [52]:
# Display validation performance for the max_depth=10 experiment

print("Decision Tree — max_depth=10")

print(f"Accuracy : {accuracy_depth10:.4f}")
print(f"Precision: {precision_depth10:.4f}")
print(f"Recall   : {recall_depth10:.4f}")
print(f"F1-score : {f1_depth10:.4f}")
print(f"ROC-AUC  : {roc_auc_depth10:.4f}")
print(f"PR-AUC   : {pr_auc_depth10:.4f}")

Decision Tree — max_depth=10
Accuracy : 0.4243
Precision: 0.1006
Recall   : 0.7626
F1-score : 0.1777
ROC-AUC  : 0.6037
PR-AUC   : 0.1139


## Phase 7.3 — Experiment 2: max_depth = 15

### Objective

Investigate whether increasing the maximum depth from 10 to 15 improves
the Decision Tree's validation performance.

In [53]:
# Create Decision Tree with maximum depth of 15
# All other settings remain the same as the previous experiment

dt_depth15 = DecisionTreeClassifier(
    max_depth=15,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [54]:
# Train the depth=15 Decision Tree using the training dataset

dt_depth15.fit(
    X_train,
    y_train
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_fea

In [55]:
# Generate class predictions for the validation dataset

y_pred_depth15 = dt_depth15.predict(
    X_validation
)

In [56]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_depth15 = dt_depth15.predict_proba(
    X_validation
)[:, 1]

In [57]:
# Calculate validation metrics for max_depth=15

accuracy_depth15 = accuracy_score(
    y_validation,
    y_pred_depth15
)

precision_depth15 = precision_score(
    y_validation,
    y_pred_depth15
)

recall_depth15 = recall_score(
    y_validation,
    y_pred_depth15
)

f1_depth15 = f1_score(
    y_validation,
    y_pred_depth15
)

roc_auc_depth15 = roc_auc_score(
    y_validation,
    y_prob_depth15
)

pr_auc_depth15 = average_precision_score(
    y_validation,
    y_prob_depth15
)

# Display validation performance for the max_depth=15 experiment

print("Decision Tree — max_depth=15")

print(f"Accuracy : {accuracy_depth15:.4f}")
print(f"Precision: {precision_depth15:.4f}")
print(f"Recall   : {recall_depth15:.4f}")
print(f"F1-score : {f1_depth15:.4f}")
print(f"ROC-AUC  : {roc_auc_depth15:.4f}")
print(f"PR-AUC   : {pr_auc_depth15:.4f}")

Decision Tree — max_depth=15
Accuracy : 0.5347
Precision: 0.1011
Recall   : 0.5963
F1-score : 0.1729
ROC-AUC  : 0.5743
PR-AUC   : 0.1066


In [58]:
# Calculate confusion matrix for the max_depth=15 experiment

cm_depth15 = confusion_matrix(
    y_validation,
    y_pred_depth15
)

print("Confusion Matrix — max_depth=15")
print(cm_depth15)

Confusion Matrix — max_depth=15
[[644919 573695]
 [ 43694  64534]]


## Phase 7.3 — Experiment 3: max_depth = 20

### Objective

Investigate whether increasing the maximum depth from 15 to 20 improves
the Decision Tree's validation performance.


In [59]:
# Create Decision Tree with maximum depth of 20
# All other settings remain the same as the previous experiments

dt_depth20 = DecisionTreeClassifier(
    max_depth=20,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [60]:
# Train the max_depth=20 Decision Tree using the training dataset

dt_depth20.fit(
    X_train,
    y_train
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_fea

In [61]:
# Generate class predictions for the validation dataset

y_pred_depth20 = dt_depth20.predict(
    X_validation
)

In [62]:
# Generate wildfire probability predictions for the validation dataset
# These probabilities are required for ROC-AUC and PR-AUC

y_prob_depth20 = dt_depth20.predict_proba(
    X_validation
)[:, 1]

In [63]:
# Calculate validation metrics for max_depth=20

accuracy_depth20 = accuracy_score(
    y_validation,
    y_pred_depth20
)

precision_depth20 = precision_score(
    y_validation,
    y_pred_depth20
)

recall_depth20 = recall_score(
    y_validation,
    y_pred_depth20
)

f1_depth20 = f1_score(
    y_validation,
    y_pred_depth20
)

roc_auc_depth20 = roc_auc_score(
    y_validation,
    y_prob_depth20
)

pr_auc_depth20 = average_precision_score(
    y_validation,
    y_prob_depth20
)


# Display validation performance for the max_depth=20 experiment

print("Decision Tree — max_depth=20")

print(f"Accuracy : {accuracy_depth20:.4f}")
print(f"Precision: {precision_depth20:.4f}")
print(f"Recall   : {recall_depth20:.4f}")
print(f"F1-score : {f1_depth20:.4f}")
print(f"ROC-AUC  : {roc_auc_depth20:.4f}")
print(f"PR-AUC   : {pr_auc_depth20:.4f}")

Decision Tree — max_depth=20
Accuracy : 0.6947
Precision: 0.1048
Recall   : 0.3638
F1-score : 0.1628
ROC-AUC  : 0.5476
PR-AUC   : 0.0976


In [64]:
# Calculate the confusion matrix for max_depth=20

cm_depth20 = confusion_matrix(
    y_validation,
    y_pred_depth20
)

print("Confusion Matrix — max_depth=20")
print(cm_depth20)

Confusion Matrix — max_depth=20
[[882408 336206]
 [ 68850  39378]]


## Phase 7.4 — Experiment: min_samples_leaf

### Objective

Investigate whether controlling the minimum number of observations allowed
in each leaf can improve the Decision Tree's validation performance.

In [65]:
# Create a Decision Tree with a minimum of 10 samples per leaf
# All other settings remain consistent with our baseline

dt_leaf10 = DecisionTreeClassifier(
    min_samples_leaf=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [66]:
# Train the min_samples_leaf=10 Decision Tree

dt_leaf10.fit(
    X_train,
    y_train
)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_

In [67]:
# Generate class predictions for the validation dataset

y_pred_leaf10 = dt_leaf10.predict(
    X_validation
)

In [68]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_leaf10 = dt_leaf10.predict_proba(
    X_validation
)[:, 1]

In [69]:
# Calculate validation metrics for min_samples_leaf=10

accuracy_leaf10 = accuracy_score(
    y_validation,
    y_pred_leaf10
)

precision_leaf10 = precision_score(
    y_validation,
    y_pred_leaf10
)

recall_leaf10 = recall_score(
    y_validation,
    y_pred_leaf10
)

f1_leaf10 = f1_score(
    y_validation,
    y_pred_leaf10
)

roc_auc_leaf10 = roc_auc_score(
    y_validation,
    y_prob_leaf10
)

pr_auc_leaf10 = average_precision_score(
    y_validation,
    y_prob_leaf10
)


# Display validation performance for min_samples_leaf=10

print("Decision Tree — min_samples_leaf=10")

print(f"Accuracy : {accuracy_leaf10:.4f}")
print(f"Precision: {precision_leaf10:.4f}")
print(f"Recall   : {recall_leaf10:.4f}")
print(f"F1-score : {f1_leaf10:.4f}")
print(f"ROC-AUC  : {roc_auc_leaf10:.4f}")
print(f"PR-AUC   : {pr_auc_leaf10:.4f}")

Decision Tree — min_samples_leaf=10
Accuracy : 0.8513
Precision: 0.1130
Recall   : 0.1202
F1-score : 0.1165
ROC-AUC  : 0.5187
PR-AUC   : 0.0869


In [70]:
# Calculate the confusion matrix for min_samples_leaf=10

cm_leaf10 = confusion_matrix(
    y_validation,
    y_pred_leaf10
)

print("Confusion Matrix — min_samples_leaf=10")
print(cm_leaf10)

Confusion Matrix — min_samples_leaf=10
[[1116562  102052]
 [  95221   13007]]


## Phase 7.4 — Experiment 2: min_samples_leaf = 5

### Objective

Investigate whether a smaller increase in min_samples_leaf can provide a
better balance between model flexibility and generalization compared with
the baseline value of 1.

In [71]:
# Create Decision Tree with a minimum of 5 samples in each leaf
# All other settings remain consistent with the baseline

dt_leaf5 = DecisionTreeClassifier(
    min_samples_leaf=5,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [72]:
# Train the min_samples_leaf=5 Decision Tree

dt_leaf5.fit(
    X_train,
    y_train
)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_f

In [73]:
# Generate class predictions for the validation dataset

y_pred_leaf5 = dt_leaf5.predict(
    X_validation
)

In [74]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_leaf5 = dt_leaf5.predict_proba(
    X_validation
)[:, 1]

In [75]:
# Calculate validation metrics for min_samples_leaf=5

accuracy_leaf5 = accuracy_score(
    y_validation,
    y_pred_leaf5
)

precision_leaf5 = precision_score(
    y_validation,
    y_pred_leaf5
)

recall_leaf5 = recall_score(
    y_validation,
    y_pred_leaf5
)

f1_leaf5 = f1_score(
    y_validation,
    y_pred_leaf5
)

roc_auc_leaf5 = roc_auc_score(
    y_validation,
    y_prob_leaf5
)

pr_auc_leaf5 = average_precision_score(
    y_validation,
    y_prob_leaf5
)


# Display validation performance for min_samples_leaf=5

print("Decision Tree — min_samples_leaf=5")

print(f"Accuracy : {accuracy_leaf5:.4f}")
print(f"Precision: {precision_leaf5:.4f}")
print(f"Recall   : {recall_leaf5:.4f}")
print(f"F1-score : {f1_leaf5:.4f}")
print(f"ROC-AUC  : {roc_auc_leaf5:.4f}")
print(f"PR-AUC   : {pr_auc_leaf5:.4f}")

Decision Tree — min_samples_leaf=5
Accuracy : 0.8744
Precision: 0.1184
Recall   : 0.0837
F1-score : 0.0980
ROC-AUC  : 0.5143
PR-AUC   : 0.0856


In [76]:
# Calculate the confusion matrix for min_samples_leaf=5

cm_leaf5 = confusion_matrix(
    y_validation,
    y_pred_leaf5
)

print("Confusion Matrix — min_samples_leaf=5")
print(cm_leaf5)

Confusion Matrix — min_samples_leaf=5
[[1151175   67439]
 [  99174    9054]]


## Phase 7.5 — Combined Hyperparameter Experiment

### Objective

Investigate whether combining the best-performing configurations from the
individual hyperparameter experiments can further improve Decision Tree
validation performance.


In [77]:
# Create a Decision Tree using both selected hyperparameter settings
# max_depth controls the maximum tree complexity
# min_samples_leaf prevents very small leaf nodes

dt_combined = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [78]:
# Train the combined Decision Tree using the training dataset

dt_combined.fit(
    X_train,
    y_train
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_fe

In [79]:
# Generate class predictions for the validation dataset

y_pred_combined = dt_combined.predict(
    X_validation
)

In [80]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_combined = dt_combined.predict_proba(
    X_validation
)[:, 1]

In [81]:
# Calculate validation metrics for the combined configuration

accuracy_combined = accuracy_score(
    y_validation,
    y_pred_combined
)

precision_combined = precision_score(
    y_validation,
    y_pred_combined
)

recall_combined = recall_score(
    y_validation,
    y_pred_combined
)

f1_combined = f1_score(
    y_validation,
    y_pred_combined
)

roc_auc_combined = roc_auc_score(
    y_validation,
    y_prob_combined
)

pr_auc_combined = average_precision_score(
    y_validation,
    y_prob_combined
)

# Display validation performance of the combined configuration

print("Decision Tree — max_depth=10 + min_samples_leaf=10")

print(f"Accuracy : {accuracy_combined:.4f}")
print(f"Precision: {precision_combined:.4f}")
print(f"Recall   : {recall_combined:.4f}")
print(f"F1-score : {f1_combined:.4f}")
print(f"ROC-AUC  : {roc_auc_combined:.4f}")
print(f"PR-AUC   : {pr_auc_combined:.4f}")

Decision Tree — max_depth=10 + min_samples_leaf=10
Accuracy : 0.4243
Precision: 0.1006
Recall   : 0.7626
F1-score : 0.1777
ROC-AUC  : 0.6037
PR-AUC   : 0.1139


In [82]:
# Calculate the confusion matrix for the combined model

cm_combined = confusion_matrix(
    y_validation,
    y_pred_combined
)

print("Confusion Matrix — Combined Model")
print(cm_combined)

Confusion Matrix — Combined Model
[[480421 738193]
 [ 25697  82531]]


## Phase 7.6 — Experiment: Entropy Splitting Criterion

### Objective

Investigate whether changing the Decision Tree splitting criterion from
Gini impurity to entropy improves validation performance.

In [83]:
# Create a Decision Tree using entropy as the splitting criterion
# All other parameters remain the same as the baseline model

dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    class_weight="balanced",
    random_state=42
)

In [84]:
# Train the entropy-based Decision Tree

dt_entropy.fit(
    X_train,
    y_train
)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(ma

In [85]:
# Generate class predictions for the validation dataset

y_pred_entropy = dt_entropy.predict(
    X_validation
)

In [86]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_entropy = dt_entropy.predict_proba(
    X_validation
)[:, 1]

In [87]:
# Calculate validation metrics for the entropy-based Decision Tree

accuracy_entropy = accuracy_score(
    y_validation,
    y_pred_entropy
)

precision_entropy = precision_score(
    y_validation,
    y_pred_entropy
)

recall_entropy = recall_score(
    y_validation,
    y_pred_entropy
)

f1_entropy = f1_score(
    y_validation,
    y_pred_entropy
)

roc_auc_entropy = roc_auc_score(
    y_validation,
    y_prob_entropy
)

pr_auc_entropy = average_precision_score(
    y_validation,
    y_prob_entropy
)

# Display validation performance for the entropy experiment

print("Decision Tree — criterion=entropy")

print(f"Accuracy : {accuracy_entropy:.4f}")
print(f"Precision: {precision_entropy:.4f}")
print(f"Recall   : {recall_entropy:.4f}")
print(f"F1-score : {f1_entropy:.4f}")
print(f"ROC-AUC  : {roc_auc_entropy:.4f}")
print(f"PR-AUC   : {pr_auc_entropy:.4f}")

Decision Tree — criterion=entropy
Accuracy : 0.8926
Precision: 0.1303
Recall   : 0.0558
F1-score : 0.0782
ROC-AUC  : 0.5114
PR-AUC   : 0.0843


In [88]:
# Calculate the confusion matrix for the entropy-based model

cm_entropy = confusion_matrix(
    y_validation,
    y_pred_entropy
)

print("Confusion Matrix — criterion=entropy")
print(cm_entropy)

Confusion Matrix — criterion=entropy
[[1178273   40341]
 [ 102186    6042]]


## Phase 7.7 — Experiment: min_samples_split

### Objective

Investigate whether changing the minimum number of training observations
required to split an internal Decision Tree node can improve validation
performance.

In [89]:
# Create a Decision Tree with min_samples_split=10
# All other parameters remain at their baseline values

dt_split10 = DecisionTreeClassifier(
    min_samples_split=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [90]:
# Train the Decision Tree using the training dataset

dt_split10.fit(
    X_train,
    y_train
)

,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_

In [91]:
# Generate class predictions for the validation dataset

y_pred_split10 = dt_split10.predict(
    X_validation
)

In [92]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_split10 = dt_split10.predict_proba(
    X_validation
)[:, 1]

In [93]:
# Calculate validation metrics for min_samples_split=10

accuracy_split10 = accuracy_score(
    y_validation,
    y_pred_split10
)

precision_split10 = precision_score(
    y_validation,
    y_pred_split10
)

recall_split10 = recall_score(
    y_validation,
    y_pred_split10
)

f1_split10 = f1_score(
    y_validation,
    y_pred_split10
)

roc_auc_split10 = roc_auc_score(
    y_validation,
    y_prob_split10
)

pr_auc_split10 = average_precision_score(
    y_validation,
    y_prob_split10
)

# Display validation performance for min_samples_split=10

print("Decision Tree — min_samples_split=10")

print(f"Accuracy : {accuracy_split10:.4f}")
print(f"Precision: {precision_split10:.4f}")
print(f"Recall   : {recall_split10:.4f}")
print(f"F1-score : {f1_split10:.4f}")
print(f"ROC-AUC  : {roc_auc_split10:.4f}")
print(f"PR-AUC   : {pr_auc_split10:.4f}")

Decision Tree — min_samples_split=10
Accuracy : 0.8867
Precision: 0.1190
Recall   : 0.0607
F1-score : 0.0804
ROC-AUC  : 0.5105
PR-AUC   : 0.0842


In [94]:
# Calculate the confusion matrix for min_samples_split=10

cm_split10 = confusion_matrix(
    y_validation,
    y_pred_split10
)

print("Confusion Matrix — min_samples_split=10")
print(cm_split10)

Confusion Matrix — min_samples_split=10
[[1169972   48642]
 [ 101655    6573]]


## Phase 7.8 — Experiment: class_weight

### Objective

Investigate whether changing the class weighting strategy can improve
Decision Tree performance for the imbalanced wildfire classification
problem.

In [95]:
# Create a Decision Tree using manual class weights
# Class 0 = No wildfire
# Class 1 = Wildfire
# Wildfire observations receive twice the weight

dt_weight2 = DecisionTreeClassifier(
    class_weight={0: 1, 1: 2},
    criterion="gini",
    random_state=42
)

In [96]:
# Train the manually weighted Decision Tree

dt_weight2.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.","{0: 1, 1: 2}"
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(m

In [97]:
# Generate class predictions for the validation dataset

y_pred_weight2 = dt_weight2.predict(
    X_validation
)

In [98]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_weight2 = dt_weight2.predict_proba(
    X_validation
)[:, 1]

In [99]:
# Calculate validation metrics for the manually weighted model

accuracy_weight2 = accuracy_score(
    y_validation,
    y_pred_weight2
)

precision_weight2 = precision_score(
    y_validation,
    y_pred_weight2
)

recall_weight2 = recall_score(
    y_validation,
    y_pred_weight2
)

f1_weight2 = f1_score(
    y_validation,
    y_pred_weight2
)

roc_auc_weight2 = roc_auc_score(
    y_validation,
    y_prob_weight2
)

pr_auc_weight2 = average_precision_score(
    y_validation,
    y_prob_weight2
)

# Display validation performance for the manual class-weight experiment

print("Decision Tree — class_weight={0:1, 1:2}")

print(f"Accuracy : {accuracy_weight2:.4f}")
print(f"Precision: {precision_weight2:.4f}")
print(f"Recall   : {recall_weight2:.4f}")
print(f"F1-score : {f1_weight2:.4f}")
print(f"ROC-AUC  : {roc_auc_weight2:.4f}")
print(f"PR-AUC   : {pr_auc_weight2:.4f}")

Decision Tree — class_weight={0:1, 1:2}
Accuracy : 0.8831
Precision: 0.1227
Recall   : 0.0704
F1-score : 0.0894
ROC-AUC  : 0.5128
PR-AUC   : 0.0844


In [100]:
# Calculate the confusion matrix for the manually weighted model

cm_weight2 = confusion_matrix(
    y_validation,
    y_pred_weight2
)

print("Confusion Matrix — class_weight={0:1, 1:2}")
print(cm_weight2)

Confusion Matrix — class_weight={0:1, 1:2}
[[1164171   54443]
 [ 100612    7616]]


## Phase 7.9 — Hyperparameter Experiment Comparison

### Objective

Consolidate the results of all Decision Tree hyperparameter experiments
and compare them with the baseline model.

In [101]:
# Create a table containing all Decision Tree hyperparameter experiments
# Baseline values are entered directly because separate baseline variables
# were not created earlier.

experiment_results = [
    {
        "Experiment": "Baseline",
        "Configuration": "Gini, balanced",
        "Accuracy": 0.8956,
        "Precision": 0.1235,
        "Recall": 0.0460,
        "F1": 0.0670,
        "ROC-AUC": 0.5085,
        "PR-AUC": 0.0834
    },

    {
        "Experiment": "Max Depth 10",
        "Configuration": "max_depth=10",
        "Accuracy": accuracy_depth10,
        "Precision": precision_depth10,
        "Recall": recall_depth10,
        "F1": f1_depth10,
        "ROC-AUC": roc_auc_depth10,
        "PR-AUC": pr_auc_depth10
    },

    {
        "Experiment": "Max Depth 15",
        "Configuration": "max_depth=15",
        "Accuracy": accuracy_depth15,
        "Precision": precision_depth15,
        "Recall": recall_depth15,
        "F1": f1_depth15,
        "ROC-AUC": roc_auc_depth15,
        "PR-AUC": pr_auc_depth15
    },

    {
        "Experiment": "Max Depth 20",
        "Configuration": "max_depth=20",
        "Accuracy": accuracy_depth20,
        "Precision": precision_depth20,
        "Recall": recall_depth20,
        "F1": f1_depth20,
        "ROC-AUC": roc_auc_depth20,
        "PR-AUC": pr_auc_depth20
    },

    {
        "Experiment": "Leaf 5",
        "Configuration": "min_samples_leaf=5",
        "Accuracy": accuracy_leaf5,
        "Precision": precision_leaf5,
        "Recall": recall_leaf5,
        "F1": f1_leaf5,
        "ROC-AUC": roc_auc_leaf5,
        "PR-AUC": pr_auc_leaf5
    },

    {
        "Experiment": "Leaf 10",
        "Configuration": "min_samples_leaf=10",
        "Accuracy": accuracy_leaf10,
        "Precision": precision_leaf10,
        "Recall": recall_leaf10,
        "F1": f1_leaf10,
        "ROC-AUC": roc_auc_leaf10,
        "PR-AUC": pr_auc_leaf10
    },

    {
        "Experiment": "Entropy",
        "Configuration": "criterion=entropy",
        "Accuracy": accuracy_entropy,
        "Precision": precision_entropy,
        "Recall": recall_entropy,
        "F1": f1_entropy,
        "ROC-AUC": roc_auc_entropy,
        "PR-AUC": pr_auc_entropy
    },

    {
        "Experiment": "Split 10",
        "Configuration": "min_samples_split=10",
        "Accuracy": accuracy_split10,
        "Precision": precision_split10,
        "Recall": recall_split10,
        "F1": f1_split10,
        "ROC-AUC": roc_auc_split10,
        "PR-AUC": pr_auc_split10
    },

    {
        "Experiment": "Class Weight 1:2",
        "Configuration": "class_weight={0:1,1:2}",
        "Accuracy": accuracy_weight2,
        "Precision": precision_weight2,
        "Recall": recall_weight2,
        "F1": f1_weight2,
        "ROC-AUC": roc_auc_weight2,
        "PR-AUC": pr_auc_weight2
    },

    {
        "Experiment": "Combined",
        "Configuration": "max_depth=10, min_samples_leaf=10",
        "Accuracy": accuracy_combined,
        "Precision": precision_combined,
        "Recall": recall_combined,
        "F1": f1_combined,
        "ROC-AUC": roc_auc_combined,
        "PR-AUC": pr_auc_combined
    }
]

# Convert the experiment results into a DataFrame
results_df = pd.DataFrame(experiment_results)

# Display the complete comparison table
results_df

,Experiment,Configuration,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Baseline,"Gini, balanced",0.895600,0.123500,0.046000,0.067000,0.508500,0.083400
1,Max Depth 10,max_depth=10,0.424297,0.100563,0.762575,0.177692,0.603727,0.113939
2,Max Depth 15,max_depth=15,0.534693,0.101114,0.596278,0.172907,0.574332,0.106565
3,Max Depth 20,max_depth=20,0.694722,0.104845,0.363843,0.162782,0.547642,0.097603
4,Leaf 5,min_samples_leaf=5,0.874429,0.118364,0.083657,0.098029,0.514330,0.085593
5,Leaf 10,min_samples_leaf=10,0.851321,0.113046,0.120181,0.116505,0.518657,0.086947
6,Entropy,criterion=entropy,0.892582,0.130263,0.055827,0.078157,0.511364,0.084308
7,Split 10,min_samples_split=10,0.886726,0.119044,0.060733,0.080432,0.510469,0.084215
8,Class Weight 1:2,"class_weight={0:1,1:2}",0.883140,0.122722,0.070370,0.089449,0.512843,0.084447
9,Combined,"max_depth=10, min_samples_leaf=10",0.424280,0.100559,0.762566,0.177686,0.603706,0.113935


## Phase 7.10 — Final Candidate Selection

### Objective

Select a Decision Tree configuration for final evaluation based on the
validation results obtained during hyperparameter optimization.

In [102]:
# Display the main metrics used when comparing the Decision Tree candidates

candidate_results = results_df[
    [
        "Experiment",
        "Configuration",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ]
]

candidate_results

,Experiment,Configuration,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Baseline,"Gini, balanced",0.895600,0.123500,0.046000,0.067000,0.508500,0.083400
1,Max Depth 10,max_depth=10,0.424297,0.100563,0.762575,0.177692,0.603727,0.113939
2,Max Depth 15,max_depth=15,0.534693,0.101114,0.596278,0.172907,0.574332,0.106565
3,Max Depth 20,max_depth=20,0.694722,0.104845,0.363843,0.162782,0.547642,0.097603
4,Leaf 5,min_samples_leaf=5,0.874429,0.118364,0.083657,0.098029,0.514330,0.085593
5,Leaf 10,min_samples_leaf=10,0.851321,0.113046,0.120181,0.116505,0.518657,0.086947
6,Entropy,criterion=entropy,0.892582,0.130263,0.055827,0.078157,0.511364,0.084308
7,Split 10,min_samples_split=10,0.886726,0.119044,0.060733,0.080432,0.510469,0.084215
8,Class Weight 1:2,"class_weight={0:1,1:2}",0.883140,0.122722,0.070370,0.089449,0.512843,0.084447
9,Combined,"max_depth=10, min_samples_leaf=10",0.424280,0.100559,0.762566,0.177686,0.603706,0.113935


## Phase 7.11 — Final Model Training and Test Evaluation

### Objective

Train the final Decision Tree configuration using the available historical
training and validation observations and evaluate its performance on the
previously untouched test dataset.

In [103]:
# Combine the training and validation input features
# to create the final dataset used for model training

X_final_train = pd.concat(
    [X_train, X_validation],
    axis=0
)

# Combine the corresponding target values

y_final_train = pd.concat(
    [y_train, y_validation],
    axis=0
)

# Check the final training dataset sizes

print("Final training features:", X_final_train.shape)
print("Final training target:", y_final_train.shape)

Final training features: (8646325, 20)
Final training target: (8646325,)


In [104]:
# Create the final Decision Tree using the selected hyperparameters

final_dt_model = DecisionTreeClassifier(
    max_depth=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

In [105]:
# Train the final Decision Tree using all available
# training and validation observations

final_dt_model.fit(
    X_final_train,
    y_final_train
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_fea

In [106]:
import pandas as pd

# Load the untouched test input features
X_test = pd.read_csv("../../data/processed/test.csv")

# Load the corresponding test target
y_test = pd.read_csv("../../data/processed/test.csv").squeeze()

# Check the test dataset
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (823955, 21)
y_test: (823955, 21)


In [107]:
# Verify that the test set contains the future time period
# that was defined during preprocessing

print(
    "Testing years:",
    X_test["year"].min(),
    "-",
    X_test["year"].max()
)

Testing years: 2023 - 2025


In [108]:
# Remove the target column from the test input features
# The model must receive exactly the same 20 features used during training

X_test = X_test.drop(
    columns=["wildfire"]
)

print("X_test shape:", X_test.shape)
print("Number of test features:", len(X_test.columns))

X_test shape: (823955, 20)
Number of test features: 20


In [109]:
# Verify that the test features are exactly the same
# as the features used when training the final model

print(
    "Features match:",
    list(X_test.columns) == list(final_dt_model.feature_names_in_)
)

Features match: True


In [110]:
# Generate final class predictions on the untouched test dataset

y_pred_test = final_dt_model.predict(
    X_test
)

In [111]:
# Generate wildfire probabilities for ROC-AUC and PR-AUC

y_prob_test = final_dt_model.predict_proba(
    X_test
)[:, 1]

In [112]:
# Extract the actual target column from the incorrectly loaded test dataset

y_test = y_test["wildfire"]

# Convert it into a one-dimensional Series
y_test = y_test.squeeze()

# Check the corrected target

print("y_test type:", type(y_test))
print("y_test shape:", y_test.shape)

print("\nTarget distribution:")
print(y_test.value_counts())

print("\nUnique values:")
print(y_test.unique())

y_test type: <class 'pandas.Series'>
y_test shape: (823955,)

Target distribution:
wildfire
0    705565
1    118390
Name: count, dtype: int64

Unique values:
[0 1]


In [113]:
# Calculate final performance metrics on the untouched test dataset

accuracy_test = accuracy_score(
    y_test,
    y_pred_test
)

precision_test = precision_score(
    y_test,
    y_pred_test
)

recall_test = recall_score(
    y_test,
    y_pred_test
)

f1_test = f1_score(
    y_test,
    y_pred_test
)

roc_auc_test = roc_auc_score(
    y_test,
    y_prob_test
)

pr_auc_test = average_precision_score(
    y_test,
    y_prob_test
)

# Display the final Decision Tree performance on the test dataset

print("FINAL DECISION TREE — TEST RESULTS")

print(f"Accuracy : {accuracy_test:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall   : {recall_test:.4f}")
print(f"F1-score : {f1_test:.4f}")
print(f"ROC-AUC  : {roc_auc_test:.4f}")
print(f"PR-AUC   : {pr_auc_test:.4f}")

FINAL DECISION TREE — TEST RESULTS
Accuracy : 0.3589
Precision: 0.1652
Recall   : 0.8538
F1-score : 0.2768
ROC-AUC  : 0.6043
PR-AUC   : 0.1872


In [114]:
# Calculate the confusion matrix on the untouched test dataset

cm_test = confusion_matrix(
    y_test,
    y_pred_test
)

print("Final Test Confusion Matrix")
print(cm_test)

Final Test Confusion Matrix
[[194674 510891]
 [ 17306 101084]]


In [115]:
# Calculate feature importance from the selected max_depth=10 Decision Tree

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": final_dt_model.feature_importances_
})

# Sort from most important to least important
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display the ranking
print("Feature Importance Ranking:")
display(feature_importance)

Feature Importance Ranking:


,Feature,Importance
0,year,0.262207
1,longitude,0.204986
2,fuel_moisture_1000hr,0.193511
3,latitude,0.184909
4,day_of_year,0.124984
5,month,0.009788
6,solar_radiation,0.007476
7,specific_humidity,0.005707
8,temperature_min,0.001801
9,temperature_max,0.001264


In [116]:
# Select the 10 most important features

top_10_features = feature_importance.head(10)["Feature"].tolist()

print("Top 10 selected features:")

for rank, feature in enumerate(top_10_features, start=1):
    importance = feature_importance.loc[
        rank - 1, "Importance"
    ]

    print(f"{rank}. {feature} → {importance:.6f}")

Top 10 selected features:
1. year → 0.262207
2. longitude → 0.204986
3. fuel_moisture_1000hr → 0.193511
4. latitude → 0.184909
5. day_of_year → 0.124984
6. month → 0.009788
7. solar_radiation → 0.007476
8. specific_humidity → 0.005707
9. temperature_min → 0.001801
10. temperature_max → 0.001264


In [117]:
# Create training data containing only the selected top 10 features

X_train_top10 = X_train[
    top_10_features
].copy()

# Create validation data containing the same top 10 features

X_validation_top10 = X_validation[
    top_10_features
].copy()

# Check the shapes

print("X_train_top10:", X_train_top10.shape)
print("X_validation_top10:", X_validation_top10.shape)

X_train_top10: (7319483, 10)
X_validation_top10: (1326842, 10)


In [118]:
from sklearn.tree import DecisionTreeClassifier

# Train a Decision Tree using only the selected top 10 features
# Keep the same hyperparameters as the selected candidate model

dt_top10 = DecisionTreeClassifier(
    max_depth=10,
    criterion="gini",
    class_weight="balanced",
    random_state=42
)

dt_top10.fit(
    X_train_top10,
    y_train
)

print("Top-10 feature Decision Tree trained successfully.")

Top-10 feature Decision Tree trained successfully.


In [119]:
# Generate class predictions for the top-10 feature model

y_pred_top10 = dt_top10.predict(
    X_validation_top10
)

# Generate probability predictions for the wildfire class
# These are required for ROC-AUC and PR-AUC

y_prob_top10 = dt_top10.predict_proba(
    X_validation_top10
)[:, 1]

print("Validation predictions generated.")

Validation predictions generated.


In [120]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Calculate metrics for the top-10 feature model

accuracy_top10 = accuracy_score(
    y_validation,
    y_pred_top10
)

precision_top10 = precision_score(
    y_validation,
    y_pred_top10
)

recall_top10 = recall_score(
    y_validation,
    y_pred_top10
)

f1_top10 = f1_score(
    y_validation,
    y_pred_top10
)

roc_auc_top10 = roc_auc_score(
    y_validation,
    y_prob_top10
)

pr_auc_top10 = average_precision_score(
    y_validation,
    y_prob_top10
)

print("TOP-10 FEATURE MODEL — VALIDATION RESULTS")
print("------------------------------------------")
print(f"Accuracy : {accuracy_top10:.4f}")
print(f"Precision: {precision_top10:.4f}")
print(f"Recall   : {recall_top10:.4f}")
print(f"F1-score : {f1_top10:.4f}")
print(f"ROC-AUC  : {roc_auc_top10:.4f}")
print(f"PR-AUC   : {pr_auc_top10:.4f}")

TOP-10 FEATURE MODEL — VALIDATION RESULTS
------------------------------------------
Accuracy : 0.4243
Precision: 0.1006
Recall   : 0.7626
F1-score : 0.1777
ROC-AUC  : 0.6032
PR-AUC   : 0.1137


In [121]:
# Store the candidate and top-10 feature model results
# so that they can be compared directly

comparison = pd.DataFrame([
    {
        "Model": "Candidate - All 20 Features",
        "Features": 20,
        "Accuracy": 0.4243,
        "Precision": 0.1006,
        "Recall": 0.7626,
        "F1": 0.1777,
        "ROC-AUC": 0.6037,
        "PR-AUC": 0.1139
    },
    {
        "Model": "Feature Selection - Top 10",
        "Features": 10,
        "Accuracy": accuracy_top10,
        "Precision": precision_top10,
        "Recall": recall_top10,
        "F1": f1_top10,
        "ROC-AUC": roc_auc_top10,
        "PR-AUC": pr_auc_top10
    }
])

display(comparison)

,Model,Features,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Candidate - All 20 Features,20,0.424300,0.100600,0.762600,0.177700,0.603700,0.11390
1,Feature Selection - Top 10,10,0.424277,0.100563,0.762612,0.177694,0.603202,0.11374
